# Unified Preprocessing Pipeline

This notebook implements a preprocessing pipeline for the arXiv dataset. Its primary goal is to standardize article identifiers, clean textual data (titles, abstracts), and integrate citation graph information into a unified JSON structure. It will be used to compare three recommendation methods. 

**Key Objectives:**
1.  **ID Normalization:** Convert all arXiv IDs (including legacy formats like `alg-geom/9202013`) into a consistent, digits-only format to ensure reliable matching across datasets.
2.  **Metadata Parsing:** Load and clean article metadata (titles, authors, categories, abstracts).
3.  **Graph Integration:** Merge the metadata with an external citation graph, resolving references to the standardized IDs.
4.  **Gold Dataset Creation:** Generate specialized evaluation datasets ("Gold Datasets") by downloading source LaTeX files and extracting ground-truth citation contexts.

## 1. Environment Setup & Configuration

We begin by importing necessary libraries and defining file paths. The configuration block allows for easy adjustment of input/output directories.

In [ ]:
import json
import gzip
import sys
import os
from typing import Optional

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

METADATA_FILE = "data/arxiv-metadata-oai-snapshot.json"
GRAPH_FILE = "data/internal-references-pdftotext.json"

OUTPUT_FILE = "data/processed/unified_articles.json"

METADATA_ID_KEY = "id"
METADATA_TITLE_KEY = "title"
METADATA_ABSTRACT_KEY = "abstract"
METADATA_AUTHORS_KEY = "authors"
METADATA_CATEGORIES_KEY = "categories"

## 2. Data Normalization Helpers

These utility functions are the core of our data cleaning strategy.

*   `normalize_to_simple_id`: Handles the complexity of arXiv's changing ID formats over the last 30+ years, stripping everything down to a purely numeric string.
*   `clean_text`: Prepares raw text (like abstracts) for NLP tasks by removing HTML tags, special characters, and normalizing whitespace.

In [ ]:
# ============================================================================
# NORMALIZATION FUNCTION 
# ============================================================================

def normalize_to_simple_id(id_str: str) -> str:
    """
    Normalize any ID format to a simple digits-only string.

    Examples:
    '9202.013' -> '9202013'
    'alg-geom/9202013' -> '9202013'
    'cs.LG/210100001' -> '210100001'
    '0001.234' -> '0001234'

    Works for both Kaggle and Graph formats.

    Args:
    id_str: ID string in Kaggle or Graph format

    Returns:
    Normalized ID containing digits only.
    """
    # Keep only digit characters to create the canonical ID used across datasets
    return ''.join(c for c in id_str if c.isdigit())

In [ ]:
# ============================================================================
# PREPROCESSING
# ============================================================================

import re

# Regex precompiled for performance
HTML_TAG_REGEX = re.compile(r"<.*?>")
SPECIAL_CHAR_REGEX = re.compile(r"[^a-zA-Z0-9\s]")

def clean_text(text: str) -> str:
    """Clean abstract text for NLP tasks"""
    if not text:
        return ""
    
    text = HTML_TAG_REGEX.sub("", text)
    text = text.replace("\\n", " ").replace("\\", "")
    text = SPECIAL_CHAR_REGEX.sub("", text)
    text = text.lower().strip()
    
    return text

## 3. Main Unified Dataset Builder

This function orchestrates the entire unification process in four steps:
1.  **Load Metadata:** Reads the massive JSONL metadata file, normalizing every ID and filtering out articles with empty abstracts.
2.  **Load Graph:** Reads the citation graph, ensuring all source and target IDs are normalized to match the metadata.
3.  **Attach Citations:** Links the two datasets. For every article in the metadata, we look up its citations in the graph, validate that the cited papers actually exist in our dataset, and attach them.
4.  **Save:** Writes the final, enriched dataset to disk.

In [ ]:
# ============================================================================
# MAIN PROCESSING
# ============================================================================

def build_unified_dataset(metadata_path: str, 
                          graph_path: str, 
                          output_path: str):
    """
    Build a unified dataset using simplified numeric-only IDs for internal consistency.
    The function reads the metadata stream, normalizes IDs, builds lookup sets,
    normalizes the citation graph IDs, attaches valid citations, and saves the final JSON.
    """
    
    print("="*80)
    print("UNIFIED PREPROCESSING - SIMPLE ID FORMAT")
    print("="*80)

    # Step 1: Load metadata and normalize IDs
    print("\n[Step 1] Loading metadata and normalizing IDs...")
    
    all_ids = set() # normalized id set
    all_articles = {} # dict keyed by normalized id
    original_id_map = {} # mapping normalized_id -> original id (for output)
    
    papers_processed = 0
    papers_kept = 0
    
    try:
        f_open = gzip.open if metadata_path.endswith('.gz') else open
        
        with f_open(metadata_path, 'rt', encoding='utf-8') as f:
            for line in f:
                try:
                    article = json.loads(line)
                    
                    # original id
                    original_id = article.get(METADATA_ID_KEY)
                    if not original_id:
                        continue
                    
                    # Normalize id
                    normalized_id = normalize_to_simple_id(original_id)
                    
                    # Add to id set
                    all_ids.add(normalized_id)
                    
                    papers_processed += 1
                    
                    if papers_processed % 100000 == 0:
                        print(f" ... {papers_processed:,} processed, {papers_kept:,} kept")
                    
                    # Extract fields
                    title = article.get(METADATA_TITLE_KEY, "")
                    abstract = article.get(METADATA_ABSTRACT_KEY, "")
                    authors = article.get(METADATA_AUTHORS_KEY, [])
                    categories = article.get(METADATA_CATEGORIES_KEY, "")
                    
                    # Filters: keep only meaningful abstracts
                    if not abstract:
                        continue
                    
                    clean_abstract = clean_text(abstract)
                    
                    if len(clean_abstract) < 50:
                        continue
                    
                    # Store using normalized ID as the key
                    all_articles[normalized_id] = {
                        "id": original_id, # keep original Kaggle ID for the output
                        "normalized_id": normalized_id, # for debugging
                        "title": title,
                        "authors": authors if isinstance(authors, list) else [],
                        "abstract": abstract,
                        "clean_text": clean_abstract,
                        "categories": categories.strip() if categories else "",
                        "refs": [] # to be filled later
                    }
                    
                    original_id_map[normalized_id] = original_id
                    papers_kept += 1
                    
                except json.JSONDecodeError:
                    # skip invalid JSON lines
                    continue
                    
        print(f"\nStep 1 finished:")
        print(f" - Papers processed: {papers_processed:,}")
        print(f" - Papers kept: {papers_kept:,}")
        print(f" - Unique normalized IDs: {len(all_ids):,}")
        
    except FileNotFoundError:
        print(f"ERROR: '{metadata_path}' not found.")
        sys.exit(1)
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)
        
    # Step 2: Load citation graph and normalize IDs
    print("\n[Step 2] Loading citation graph and normalizing IDs...")
    
    try:
        f_open = gzip.open if graph_path.endswith('.gz') else open
        
        print(" Loading graph file...")
        with f_open(graph_path, 'rt', encoding='utf-8') as f:
            citation_graph_raw = json.load(f)
            
        print(f" Graph loaded: {len(citation_graph_raw):,} papers")
        
        # Normalize all ids in the graph (source and targets)
        print(" Normalizing graph IDs...")
        citation_graph_normalized = {}
        
        for source_id_raw, refs_raw in citation_graph_raw.items():
            source_id_norm = normalize_to_simple_id(source_id_raw)
            
            refs_normalized = []
            if isinstance(refs_raw, list):
                refs_normalized = [normalize_to_simple_id(ref) for ref in refs_raw]
            
            citation_graph_normalized[source_id_norm] = refs_normalized
            
        print(f" {len(citation_graph_normalized):,} normalized IDs in graph")
        
    except FileNotFoundError:
        print(f"ERROR: '{graph_path}' not found.")
        sys.exit(1)
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)
        
    # Step 3: Attach citations (simple match using normalized ids)
    print("\n[Step 3] Adding citations...")
    
    refs_added = 0
    refs_total = 0
    refs_valid = 0
    
    papers_checked = 0
    
    for normalized_id, article_data in all_articles.items():
        papers_checked += 1
        
        if papers_checked % 50000 == 0:
            print(f" ... {papers_checked:,}/{len(all_articles):,} papers")
        
        # direct lookup in the normalized graph
        if normalized_id in citation_graph_normalized:
            refs_normalized = citation_graph_normalized[normalized_id]
            
            refs_total += len(refs_normalized)
            
            # keep only refs that exist in metadata (avoid dangling refs)
            valid_refs = [ref for ref in refs_normalized if ref in all_ids]
            
            refs_valid += len(valid_refs)
            
            # deduplicate
            unique_refs = list(set(valid_refs))
            
            # map back to original Kaggle IDs for output
            original_refs = [original_id_map.get(ref, ref) for ref in unique_refs]
            
            article_data["refs"] = original_refs
            
            if original_refs:
                refs_added += 1
                
    print(f"\nStep 3 finished:")
    print(f" - Papers with citations: {refs_added:,}")
    print(f" - Total references (raw): {refs_total:,}")
    print(f" - Valid references: {refs_valid:,}")
    
    if len(all_articles) > 0:
        coverage = (refs_added / len(all_articles)) * 100
        print(f" - Coverage: {coverage:.1f}%")
        
    # Step 4: Save output JSON
    print("\n[Step 4] Saving output...")
    
    try:
        output_dir = os.path.dirname(output_path)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)
            
        # Remove debug field 'normalized_id' before saving
        for article in all_articles.values():
            article.pop('normalized_id', None)
            
        final_articles = list(all_articles.values())
        
        stats = {
            'papers_with_refs': sum(1 for a in final_articles if a["refs"]),
            'papers_without_refs': sum(1 for a in final_articles if not a["refs"]),
            'total_refs': sum(len(a["refs"]) for a in final_articles)
        }
        
        final_data = {
            "articles": final_articles,
            "metadata": {
                "total_papers": len(final_articles),
                "papers_with_refs": stats['papers_with_refs'],
                "papers_without_refs": stats['papers_without_refs'],
                "total_references": stats['total_refs'],
                "has_clean_text": True,
                "has_citations": True,
                "has_categories": True,
                "id_format": "Kaggle original format",
                "normalization": "digits only internally"
            }
        }
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(final_data, f, indent=2)
            
        print(f"\nDONE. Dataset saved to: {output_path}")
        print(f"\nFinal statistics:")
        print(f" - Total papers: {len(final_articles):,}")
        print(f" - Papers with references: {stats['papers_with_refs']:,}")
        print(f" - Papers without references: {stats['papers_without_refs']:,}")
        print(f" - Total references: {stats['total_refs']:,}")
        
        if stats['papers_with_refs'] > 0:
            avg_refs = stats['total_refs'] / stats['papers_with_refs']
            print(f" - Avg refs per paper: {avg_refs:.2f}")
            
    except Exception as e:
        print(f"ERROR while saving: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)

### Execution
Run the unified dataset builder with the paths configured above.

In [ ]:
build_unified_dataset(
    METADATA_FILE,
    GRAPH_FILE,
    OUTPUT_FILE
)

# Gold Dataset Creation & Subdataset Cleaning Pipeline

This section implements the end-to-end pipeline for generating "Gold Datasets"—subsets of articles where citations are explicitly linked to their textual context using full LaTeX source files.

**Pipeline Steps:**
1.  **Load** the unified arXiv dataset created above.
2.  **Generate Candidates** using three distinct strategies: Water-filling (Stratified), Most Cited (Top-N), and Quartile-based.
3.  **Create Gold Datasets**: For each strategy, download LaTeX source files, resolve citation keys (e.g., `\cite{alexnet}`) to arXiv IDs, and extract the surrounding text.
4.  **Clean Subdatasets**: Produce final training datasets by removing the Gold articles to prevent data leakage.
5.  **Save** all artifacts in structured JSON.

In [3]:
import json
import os
import re
import tarfile
import gzip
import shutil
import requests
from typing import List, Dict, Set, Any, Tuple, Optional
from collections import defaultdict
from difflib import SequenceMatcher

## 1. Gold Dataset Configuration

Setup of input/output paths and regex patterns used to parse LaTeX files. `TARGET_GOLD_SIZE` determines how many articles with resolved citations we aim to collect per strategy.

In [7]:
DATA_PATH = 'data/processed/unified_articles.json'
OUTPUT_DIR = 'data/gold_datasets_linked/'
TEMP_DIR = 'temp_latex_downloads_linked/'
TARGET_GOLD_SIZE = 20
CONTEXT_WINDOW = 400

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
if not os.path.exists(TEMP_DIR):
    os.makedirs(TEMP_DIR)

# --- Pre-compiled Regex ---
# Matches LaTeX citations like \cite{foo}, \citep{foo, bar}
CITE_PATTERN = re.compile(r'(\\(cite|citep|citet|bibcite)\{([^}]+)\})')

# Matches bibliography items: \bibitem{key} The Title ...
BIBITEM_PATTERN = re.compile(r'\\bibitem\{([^}]+)\}([\s\S]*?)(?=\\bibitem|\\end\{thebibliography\})')

def clean_latex_text(text: str) -> str:
    """Cleans LaTeX formatting for readability."""
    if not text: return ""
    # Remove comments
    text = re.sub(r'%.*', '', text)
    # Simplify common commands
    text = re.sub(r'\\(textbf|textit|emph|section|subsection)\{([^}]+)\}', r'\2', text)
    text = re.sub(r'\$([^$]+)\$', r'\1', text) # Inline math
    # Normalize whitespace
    return re.sub(r'\s+', ' ', text).strip()

## 2. Data Indexing

We build a global index of `ID -> Normalized Title`. This is crucial for the fuzzy matching step, where we check if a title found in a paper's LaTeX bibliography corresponds to a known arXiv ID in our dataset.

In [8]:
def load_and_index_data(filepath: str) -> Tuple[List[Dict], Dict[str, str]]:
    """
    Loads data and creates a global mapping of ID -> Title.
    This allows us to look up the titles of referenced IDs.
    """
    print(f"Loading dataset from {filepath}...")
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    articles = data['articles'] if isinstance(data, dict) and 'articles' in data else data
    
    print("Building ID -> Title index...")
    id_to_title = {}
    for art in articles:
        if 'id' in art and 'title' in art:
            # Normalize title for better matching (lowercase, no punctuation)
            clean_t = re.sub(r'[^a-z0-9\s]', '', art['title'].lower())
            id_to_title[art['id']] = clean_t
            
    print(f"Indexed {len(id_to_title)} titles.")
    return articles, id_to_title

articles_data, global_id_to_title = load_and_index_data(DATA_PATH)

## 3. Source File Extraction

These functions handle the retrieval and extraction of raw source files from arXiv.
*   `download_source`: Downloads the `.tar.gz` package.
*   `extract_source_content`: Unpacks the archive and concatenates all `.tex` files into a single body string. It also separates the bibliography (either from `.bbl` files or embedded `\thebibliography`).

In [9]:
def download_source(arxiv_id: str, save_dir: str) -> Optional[str]:
    """Downloads .tar.gz source from arXiv."""
    url = f"https://arxiv.org/e-print/{arxiv_id}"
    save_path = os.path.join(save_dir, f"{arxiv_id}.tar.gz")
    try:
        response = requests.get(url, stream=True, timeout=20)
        if response.status_code == 200 and 'pdf' not in response.headers.get('content-type', ''):
            with open(save_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            return save_path
    except Exception as e:
        print(f"Error downloading {arxiv_id}: {e}")
        return None

def extract_source_content(file_path: str) -> Tuple[str, str]:
    """
    Extracts:
    1. Full combined LaTeX body (for text extraction)
    2. Combined Bibliography content (from .bbl files or embedded \thebibliography)
    """
    full_body = ""
    full_bib = ""
    
    try:
        if tarfile.is_tarfile(file_path):
            with tarfile.open(file_path, 'r:*') as tar:
                for member in tar.getmembers():
                    # Extract Text
                    if member.name.endswith('.tex'):
                        try:
                            f = tar.extractfile(member)
                            content = f.read().decode('utf-8', errors='ignore')
                            full_body += f"\n% --- FILE: {member.name} ---\n{content}"
                            # Check for embedded bibliography
                            if '\\begin{thebibliography}' in content:
                                full_bib += content
                        except: pass
                    
                    # Extract Bibliography file
                    elif member.name.endswith('.bbl'):
                        try:
                            f = tar.extractfile(member)
                            full_bib += f.read().decode('utf-8', errors='ignore') + "\n"
                        except: pass
        else:
            # Single file case
            with gzip.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
                content = f.read()
                full_body = content
                full_bib = content
                
    except Exception as e:
        print(f"Error extracting {file_path}: {e}")
        
    return full_body, full_bib

## 4. Citation Resolution Logic

This component performs the "Linkage" in our pipeline.
1.  **Parse:** We read the extracted bibliography to find pairs of `(Citation Key, Paper Title)`. E.g., `('alexnet', 'ImageNet Classification with Deep CNNs...')`.
2.  **Resolve:** We take the list of actual arXiv IDs that the metadata says this paper cites. We fuzzy-match the parsed titles against the titles of these known IDs. If a match is strong (>0.65), we map the local key `alexnet` to the global ID `1207.0560`.

In [11]:
def parse_bibliography_keys(bib_content: str) -> Dict[str, str]:
    """
    Parses \bibitem{key} ... entries.
    Returns map: { cite_key: raw_bib_text }
    """
    bib_map = {}
    for match in BIBITEM_PATTERN.finditer(bib_content):
        key = match.group(1).strip()
        text = match.group(2)
        # Clean up the text to find a title
        # We assume the title is the longest segment or use a simple heuristic
        clean_text = re.sub(r'\\(newblock|emph|textit|textbf)', ' ', text)
        clean_text = re.sub(r'[^a-zA-Z0-9\s]', '', clean_text.lower())
        bib_map[key] = clean_text
    return bib_map

def resolve_citations(bib_map: Dict[str, str], known_ref_ids: List[str]) -> Dict[str, str]:
    """
    Links citation keys to real ArXiv IDs using fuzzy title matching.
    
    Args:
    bib_map: {cite_key: bib_text_from_latex}
    known_ref_ids: List of ArXiv IDs that this paper actually cites (from dataset)
    
    Returns:
    key_to_id: {cite_key: arxiv_id}
    """
    key_to_id = {}
    
    # 1. Retrieve titles for known reference IDs
    candidate_titles = {}
    for rid in known_ref_ids:
        if rid in global_id_to_title:
            candidate_titles[rid] = global_id_to_title[rid]

    if not candidate_titles:
        return {}

    # 2. Match keys to IDs
    for key, bib_text in bib_map.items():
        best_score = 0.0
        best_id = None
        
        # Optimization: Bib text is usually long. Title is a substring.
        # We check if the candidate title exists roughly inside the bib text.
        for rid, title in candidate_titles.items():
            # Fast check: is title subset of bib text?
            if title in bib_text:
                score = 1.0
            else:
                # Slow check: SequenceMatcher
                # We only match first 200 chars of bib entry to save time
                score = SequenceMatcher(None, title, bib_text[:300]).ratio()
            
            if score > best_score:
                best_score = score
                best_id = rid
        
        # Threshold for a valid match
        if best_score > 0.65: 
            key_to_id[key] = best_id
            
    return key_to_id

## 5. Subdataset Generation Strategies

We define three methods to select candidate articles for our subdatasets. All methods return a dictionary wrapper `{"articles": [...]}`.

1.  **Most Cited:** Simple Top-N based on reference count.
2.  **Stratified (Simple):** Uniform sampling across categories.
3.  **Balanced Quartiles:** Ensures representation from all tiers of citation impact (Top 25% to Bottom 25%).
4.  **Water Filling (Cascade):** A robust stratified approach that handles uneven category sizes by filling quotas from larger categories when smaller ones run out.

In [12]:
def get_candidates_most_cited(articles: List[Dict], top_n: int = 50000) -> List[Dict]:
    """Returns top N articles sorted by number of references (or citations if available)."""
    # Sorting by number of OUTGOING references as a proxy if citation count isn't in metadata.
    # If you have an 'citations_count' field, use that instead.
    # Assuming 'refs' field exists from preprocessing.
    sorted_articles = sorted(articles, key=lambda x: len(x.get('refs', [])), reverse=True)
    return sorted_articles[:top_n]

In [13]:
def get_candidates_stratified(articles: List[Dict], total_k: int = 50000) -> List[Dict]:
    """Returns K articles distributed across categories (Water Filling)."""
    # 1. Group by category
    cat_map = defaultdict(list)
    for art in articles:
        cats = art.get('categories', '').split()
        if cats:
            primary_cat = cats[0]
            cat_map[primary_cat].append(art)
            
    # 2. Calculate target per category
    n_cats = len(cat_map)
    if n_cats == 0: return []
    target_per_cat = max(1, total_k // n_cats)
    
    selected = []
    # 3. Select top cited from each category
    for cat, arts in cat_map.items():
        # Sort by refs count
        sorted_arts = sorted(arts, key=lambda x: len(x.get('refs', [])), reverse=True)
        selected.extend(sorted_arts[:target_per_cat])
        
    return selected[:total_k]

In [14]:
def get_candidates_balanced_quartiles(articles: List[Dict], total_n: int = 50000) -> List[Dict]:
    """
    Creates a subdataset of size `total_n` composed of equal shares from each 
    citation quartile of the original dataset.
    
    - 25% of `total_n` from Q1 (Top 25% most cited)
    - 25% of `total_n` from Q2
    - 25% of `total_n` from Q3
    - 25% of `total_n` from Q4 (Bottom 25% least cited)
    """
    # 1. Sort all articles by citation count (Descending: Most cited -> Least cited)
    # Using 'refs' length as proxy for citations if 'citations_count' is unavailable
    sorted_articles = sorted(articles, key=lambda x: len(x.get('refs', [])), reverse=True)
    
    total_source = len(sorted_articles)
    if total_source == 0:
        return []

    # 2. Define the size of one quartile in the SOURCE dataset
    source_q_size = total_source // 4
    
    # 3. Define how many items we want from each quartile in the OUTPUT
    target_per_quartile = total_n // 4
    
    balanced_selection = []
    
    # 4. Iterate through the 4 quartiles
    for q in range(4):
        # Define start/end indices for this quartile in the sorted source list
        start_idx = q * source_q_size
        
        # For the last quartile, ensure we go to the very end (handle remainder)
        if q == 3:
            end_idx = total_source
        else:
            end_idx = start_idx + source_q_size
        
        # Extract the full pool for this quartile
        quartile_pool = sorted_articles[start_idx:end_idx]
        
        # Select the top portion of this specific quartile to meet our target
        # (Since pool is sorted, this picks the 'best' of the quartile. 
        # Use random.sample(quartile_pool, target_per_quartile) if you want random sampling within the quartile)
        selection = quartile_pool[:target_per_quartile]
        
        balanced_selection.extend(selection)
        
    # 5. Handle Rounding Errors (if total_n isn't divisible by 4)
    # Fill any remaining slots with the highest cited papers not yet selected (from Q1)
    missing = total_n - len(balanced_selection)
    if missing > 0:
        # We took the first 'target_per_quartile' from Q1. 
        # The next best candidates start immediately after that index.
        remainder_start = target_per_quartile
        remainder_end = remainder_start + missing
        
        # Ensure we don't go out of bounds of Q1
        if remainder_end < source_q_size:
            balanced_selection.extend(sorted_articles[remainder_start:remainder_end])
    
    return balanced_selection

In [15]:
from typing import List, Dict, Any, Set
from collections import defaultdict
import networkx as nx
import math

def get_candidates_stratified_waterfilling(articles: List[Dict], top_n: int = 50000) -> Dict[str, List[Dict]]:
    """
    Selects K articles using a Water Filling (Cascade) approach to balance categories.
    Prioritizes highly cited articles within each category (using in-degree).
    
    Returns:
    Dict with structure {"articles": [selected_article_dicts]}
    """
    print(f"Starting Cascade Sampling (Target: {top_n} articles)...")
    
    # 1. Create the graph and the Primary Category dictionary
    print(f"Building graph and mapping primary categories for {len(articles)} articles...")
    
    primary_category_dict: Dict[str, List[str]] = {}
    G_full = nx.DiGraph()
    article_id_to_data = {} 
    
    for article in articles:
        article_id = article.get('id')
        
        if article_id is not None:
            # Extract primary category
            cat_string = article.get('categories', '')
            article_cats = cat_string.split(" ") if cat_string else []
            primary_cat = article_cats[0] if article_cats else "unknown"
            
            # Build Graph for scoring (in-degree calculation)
            G_full.add_node(article_id) 
            article_id_to_data[article_id] = article
            for link in article.get('refs', []):
                G_full.add_edge(article_id, link)
            
            # Map article to its PRIMARY category
            if primary_cat not in primary_category_dict:
                primary_category_dict[primary_cat] = []
            primary_category_dict[primary_cat].append(article_id)

    # 2. Prepare for Selection
    final_node_ids = set()
    
    # Sort categories by size (largest first)
    sorted_categories = sorted(
        primary_category_dict.keys(), 
        key=lambda k: len(primary_category_dict[k]), 
        reverse=True
    )
    
    nb_categories = len(sorted_categories)
    if nb_categories == 0:
        return {"articles": []}

    # Calculate initial target per category
    target_per_cat = int(math.ceil(top_n / nb_categories))
    print(f"Targeting approx. {target_per_cat} articles per category across {nb_categories} categories...")

    # Pre-calculate degrees for O(1) access
    in_degrees = dict(G_full.in_degree())
    
    # Store sorted articles per category to avoid re-sorting in Pass 2
    sorted_articles_map = {} 

    # --- PASS 1: The "Fair Share" Allocation ---
    # Take up to 'target_per_cat' from each category, sorted by quality (in-degree)
    for cat in sorted_categories:
        article_ids = primary_category_dict[cat]
        
        # Sort by in-degree (highest first)
        sorted_articles = sorted(
            article_ids, 
            key=lambda x: in_degrees.get(x, 0), 
            reverse=True
        )
        sorted_articles_map[cat] = sorted_articles # Save for Pass 2
        
        # Take the quota
        take_count = min(target_per_cat, len(sorted_articles))
        selected_subset = sorted_articles[:take_count]
        
        final_node_ids.update(selected_subset)
        
    print(f"After Pass 1: Selected {len(final_node_ids)} articles.")

    # --- PASS 2: The "Fill Remaining" (Water Filling) ---
    # If we haven't reached top_n (because some categories were small), fill from the big ones
    if len(final_node_ids) < top_n:
        remaining_needed = top_n - len(final_node_ids)
        print(f"Pass 2: Filling gap of {remaining_needed} articles from remaining pools...")
        
        for cat in sorted_categories:
            if remaining_needed <= 0:
                break
            
            all_cat_articles = sorted_articles_map[cat]
            
            # Identify articles NOT yet selected
            # We already took the first 'take_count' (calculated above)
            already_taken_count = min(target_per_cat, len(all_cat_articles))
            
            candidates = all_cat_articles[already_taken_count:]
            
            if not candidates:
                continue
            
            # Take as many as needed or available
            take_extra = min(remaining_needed, len(candidates))
            
            final_node_ids.update(candidates[:take_extra])
            remaining_needed -= take_extra

    print(f"Total unique articles selected: {len(final_node_ids)}")

    # 3. Return Articles
    print("Finalizing candidate list...")
    
    filtered_articles = []
    for node_id in final_node_ids:
        if node_id in article_id_to_data:
            filtered_articles.append(article_id_to_data[node_id])

    return {"articles": filtered_articles}

## 6. Processing Pipeline Execution

This block ties everything together. It defines `process_article_for_gold` (the per-article worker) and `run_gold_creation_pipeline` (the manager that loops through strategies).

1.  **Candidates:** Wrapper dictionary is generated.
2.  **Save Subdataset:** The full set of candidates is saved immediately.
3.  **Gold Extraction:** We iterate through the candidates to find valid Gold articles.
4.  **Save Gold:** The results are saved separately.

In [19]:
def process_article_for_gold(article: Dict) -> Optional[Dict]:
    """
    Full pipeline for a single article.
    Returns dict {id: ..., chunks: {ref_id: [text, text]}} if successful.
    """
    aid = article['id']
    known_refs = article.get('refs', [])
    if not known_refs:
        return None

    # 1. Download
    source_path = download_source(aid, TEMP_DIR)
    if not source_path: return None
    
    # 2. Extract
    tex_body, bib_content = extract_source_content(source_path)
    
    # 3. Parse & Resolve Keys
    bib_keys_map = parse_bibliography_keys(bib_content)
    key_to_arxiv_id = resolve_citations(bib_keys_map, known_refs)
    
    if not key_to_arxiv_id:
        # Cleanup
        if os.path.exists(source_path): os.remove(source_path)
        return None
    
    # 4. Extract Chunks linked to IDs
    # Structure: { ref_id: [chunk1, chunk2] }
    chunks_by_id = defaultdict(list)
    
    for match in CITE_PATTERN.finditer(tex_body):
        cite_keys_str = match.group(3)
        # Handle multiple keys: \cite{key1, key2}
        keys = [k.strip() for k in cite_keys_str.split(',')]
        
        # Get context
        start, end = match.span()
        context_start = max(0, start - CONTEXT_WINDOW)
        context_end = min(len(tex_body), end + CONTEXT_WINDOW)
        raw_text = tex_body[context_start:context_end]
        clean_text = clean_latex_text(raw_text)
        
        # Assign this text to every valid ID found in the keys
        for k in keys:
            if k in key_to_arxiv_id:
                target_id = key_to_arxiv_id[k]
                chunks_by_id[target_id].append(clean_text)
                
    # Cleanup
    if os.path.exists(source_path): os.remove(source_path)
    
    if chunks_by_id:
        return {
            "id": aid,
            "citations": dict(chunks_by_id) # Convert defaultdict to dict for JSON
        }
    return None

def run_gold_creation_pipeline(strategies: Dict, articles_data: List[Dict]):
    """
    Executes the pipeline for multiple strategies.
    Expects 'strat_func' to return a dict: {"articles": [list_of_articles]}
    """
    for strat_name, strat_func in strategies.items():
        print(f"\n=== Running strategy: {strat_name} ===")
        
        # 1. Generate Candidates (Returns Dict {'articles': [...]})
        candidates_wrapper = strat_func(articles_data)
        
        # 2. Save Full Subdataset (Preserves structure {"articles": ...})
        subdataset_path = os.path.join(OUTPUT_DIR, f'subdataset_{strat_name}.json')
        with open(subdataset_path, 'w', encoding='utf-8') as f:
            json.dump(candidates_wrapper, f, indent=2)
            
        # Extract the list for processing
        candidate_list = candidates_wrapper.get("articles", [])
        print(f"Saving {len(candidate_list)} candidate articles to {subdataset_path}")
        
        # 3. Create Gold Dataset
        gold_dataset = []
        gold_ids_to_remove = []
        
        print(f"Processing candidates to find {TARGET_GOLD_SIZE} valid Gold articles...")
        
        for art in candidate_list:
            if len(gold_dataset) >= TARGET_GOLD_SIZE:
                break
                
            result = process_article_for_gold(art)
            if result:
                gold_dataset.append(result)
                gold_ids_to_remove.append(art['id'])
                print(f"✅ Added {art['id']} ({len(result['citations'])} resolved refs)")
            else:
                # Optional: Print less frequency to reduce clutter
                print(f"❌ Skipped {art['id']} (No parsed refs)")
                
        # 4. Save Gold Dataset
        gold_path = os.path.join(OUTPUT_DIR, f'gold_dataset_{strat_name}.json')
        with open(gold_path, 'w', encoding='utf-8') as f:
            json.dump(gold_dataset, f, indent=2)
            
        print(f"Done with {strat_name}. Gold dataset saved to {gold_path}")

### Select Strategies
Uncomment the strategies you wish to run.

In [20]:
strategies = {
    # "quartiles": lambda data: get_candidates_balanced_quartiles(data, total_n=50000),
    # "most_cited": lambda data: get_candidates_most_cited(data, top_n=50000),
    # "stratified": lambda data: get_candidates_stratified(data, total_k=50000),
    "waterfilling": lambda data: get_candidates_stratified_waterfilling(data, top_n=50000),
}

In [21]:
# ==========================================
# EXECUTION PHASE
# ==========================================
run_gold_creation_pipeline(strategies, articles_data)

## 7. Subdataset Cleaning & Finalization

Once the Gold Datasets are created, we must remove those articles from the main subdatasets to ensure our training/test split is clean. The following two blocks handle:

1.  **Cleaning Subdatasets:** Removing Gold IDs from the `subdataset_*.json` files.
2.  **Cleaning Gold Citations:** Stripping the actual `\cite{...}` strings from the gold dataset text to prevent the model from trivially finding citations.

In [27]:
# Erasing articles used in gold from each subdataset:
for strat_name in strategies.keys():
    print(f"\n=== Cleaning subdataset for strategy: {strat_name} ===")
    
    # 1. Load gold dataset
    gold_path = os.path.join(OUTPUT_DIR, f'gold_dataset_{strat_name}.json')
    if not os.path.exists(gold_path):
        print(f"Gold dataset {gold_path} not found, skipping.")
        continue
        
    with open(gold_path, 'r', encoding='utf-8') as f:
        gold_data = json.load(f)
    gold_ids = set([art['id'] for art in gold_data])
    
    # 2. Load subdataset (Structure: {"articles": [...]})
    subdataset_path = os.path.join(OUTPUT_DIR, f'subdataset_{strat_name}.json')
    if not os.path.exists(subdataset_path):
        print(f"Subdataset {subdataset_path} not found, skipping.")
        continue

    with open(subdataset_path, 'r', encoding='utf-8') as f:
        subdataset_data = json.load(f)
        
    original_articles = subdataset_data.get('articles', [])
    
    # 3. Filter out gold articles
    clean_article_list = [art for art in original_articles if art['id'] not in gold_ids]
    
    # 4. Structure the output to match input (Dict wrapper)
    clean_output_data = {
        "articles": clean_article_list
    }
    
    # 5. Save cleaned subdataset
    clean_path = os.path.join(OUTPUT_DIR, f'clean_subdataset_{strat_name}.json')
    with open(clean_path, 'w', encoding='utf-8') as f:
        json.dump(clean_output_data, f, indent=2)
    
    print(f"Cleaned subdataset saved to {clean_path}")
    print(f"Size reduced from {len(original_articles)} to {len(clean_article_list)} articles.")

In [28]:

# In each gold dataset, we remove any string like \cite{...} to avoid leakage.
for strat_name in strategies.keys():
    print(f"\n=== Cleaning citations in gold dataset for strategy: {strat_name} ===")
    gold_path = os.path.join(OUTPUT_DIR, f'gold_dataset_{strat_name}.json')
    if not os.path.exists(gold_path):
        print(f"Gold dataset {gold_path} not found, skipping.")
        continue
    with open(gold_path, 'r') as f:
        gold_data = json.load(f)
    
    # Clean citations in chunks
    for art in gold_data:
        for ref_id, chunks in art['citations'].items():
            cleaned_chunks = []
            for chunk in chunks:
                cleaned_chunk = CITE_PATTERN.sub('', chunk)
                cleaned_chunks.append(cleaned_chunk)
            art['citations'][ref_id] = cleaned_chunks
            
    # Save cleaned gold dataset
    cleaned_gold_path = os.path.join(OUTPUT_DIR, f'gold_dataset_cleaned_{strat_name}.json')
    with open(cleaned_gold_path, 'w') as f:
        json.dump(gold_data, f, indent=2)
        
    print(f"Cleaned gold dataset saved to {cleaned_gold_path}.")

In [ ]:
# Cleanup Temp Directory
shutil.rmtree(TEMP_DIR, ignore_errors=True)
print("\nAll tasks completed. Temporary files removed.")